# RUKOPYS Submission: YOLO + Qwen3-VL Ukrainian Curriculum Stage 2

Kaggle inference notebook: YOLO detects `bbox`/`type`; Qwen3-VL 8B Instruct performs OCR with the Ukrainian curriculum Stage 2 LoRA adapter. PP-StructureV3/PaddleOCR is disabled by default to reduce runtime and installation risk.


In [ ]:
# =========================
# CONFIG - edit this cell
# =========================

INSTALL_DEPS = True
INSTALL_DOCLAYOUT_YOLO = True
INSTALL_PADDLEOCR = False
INSTALL_TRANSFORMERS_FROM_GIT = True

DOCLAYOUT_REPO = '/kaggle/working/DocLayout-YOLO'
PADDLE_INSTALL_MODE = 'skip'  # choices: 'cpu', 'gpu', 'skip'
PADDLEPADDLE_CPU_PACKAGE = 'paddlepaddle==3.2.2'
PADDLEPADDLE_GPU_PACKAGE = 'paddlepaddle-gpu==3.2.2'
PADDLE_CPU_INDEX_URL = 'https://www.paddlepaddle.org.cn/packages/stable/cpu/'
PADDLE_GPU_INDEX_URL = 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'
PADDLEOCR_PACKAGE = 'paddleocr[all]>=3.2.0'
PADDLE_PDX_MODEL_SOURCE = ''  # '', 'HUGGINGFACE', or 'BOS'

# Add your Kaggle input paths here. The first existing local path wins; the HF id is used if internet is enabled.
BASE_MODEL_CANDIDATES = [
    '/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1',
    '/kaggle/input/qwen3-vl-8b-instruct',
    'Qwen/Qwen3-VL-8B-Instruct',
]

# Ukrainian curriculum final OCR adapter from 02_stage2_final_format_finetune.ipynb.
USE_QWEN_LORA = True
LORA_CANDIDATES = [
    # Common Kaggle input layouts. Keep the first existing path.
    '/kaggle/input/qwen3vl-ukrainian-stage2-final-format/qwen3vl_ukrainian_stage2_final_lora_final',
    '/kaggle/input/qwen3vl-ukrainian-stage2-final-format/qwen3vl_ukrainian_stage2_final_format/qwen3vl_ukrainian_stage2_final_lora_final',
    '/kaggle/input/qwen3vl-ukrainian-stage2-final/qwen3vl_ukrainian_stage2_final_lora_final',
    '/kaggle/input/ukrainian-curriculum-stage2/qwen3vl_ukrainian_stage2_final_lora_final',
    '/kaggle/input/ukrainian-curriculum-stage2/qwen3vl_ukrainian_stage2_final_format/qwen3vl_ukrainian_stage2_final_lora_final',
    '/kaggle/input/qwen3vl-ukrainian-curriculum-stage2/qwen3vl_ukrainian_stage2_final_lora_final',
    '/kaggle/working/qwen3vl_ukrainian_stage2_final_format/qwen3vl_ukrainian_stage2_final_lora_final',
]
LORA_FALLBACK_SEARCH = True
LOAD_LORA_CROP_PROMPTS = True  # match prompts saved by the Stage 2 adapter

# Add your YOLO .pt path here.
YOLO_WEIGHT_CANDIDATES = [
    '/kaggle/input/your-yolo-weights/model.pt',
    '/kaggle/working/model.pt',
]

# Expected: DATASET_ROOT/test/metadata.jsonl and DATASET_ROOT/test/images/*
DATASET_ROOT = '/kaggle/input/datasets/quii29/rukopys-dataset'
RUN_SPLIT = 'test'  # choices: 'test', 'validation'
OUTPUT_CSV = 'submission.csv'
TEST_MODE = False
TEST_LIMIT = 4
RESUME_OUTPUT = True
CHECKPOINT_EVERY = 10
MULTI_GPU = True
GPU_IDS = [0, 1]
PARTIAL_PREFIX = 'yolo_qwen_ukr_stage2_no_pp_partial_gpu'
VERBOSE_LOG = False
LOG_IMAGE_DONE_EVERY = 1
LOG_TABLE_HINT_PREVIEW = False
PP_HINT_PREVIEW_CHARS = 300

# Qwen settings.
QWEN_DEVICE = 'auto'  # 'auto', 'cuda:0', or 'cpu'
QWEN_LOAD_IN_4BIT = True
QWEN_ATTN_IMPLEMENTATION = 'sdpa'
CROP_OCR_MODE = 'all_text'  # choices: 'none', 'smart', 'all_text'
CROP_BATCH_SIZE = 1
MAX_PIXELS_CROP = 262_144
MAX_PIXELS_TABLE_QWEN = 700_000
MAX_NEW_TOKENS_CROP = 192
MAX_NEW_TOKENS_TABLE = 512
MAX_REGION_TEXT_CHARS = 4000
CROP_PAD_RATIO = 0.04

# YOLO settings.
YOLO_BACKEND = 'auto'  # choices: 'auto', 'doclayout_yolo', 'ultralytics'
YOLO_DEVICE = 'auto'  # 'auto', 0, '0', 'cuda:0', or 'cpu'
YOLO_IMG_SIZE = 1280
YOLO_CONF = 0.20
YOLO_MAX_DET = 220
YOLO_IOU_NMS = 0.60
YOLO_DEDUP_IOU = 0.90
YOLO_PAD_SCALE_X = 0.00
YOLO_PAD_SCALE_Y = 0.00

# PP-StructureV3 table helper. Disabled by default; enable only if table hints improve validation.
PP_ENABLE = False
PP_DEVICE = 'cpu'  # examples: 'cpu', 'gpu:0'
PP_LANG = 'uk'
PP_USE_DOC_ORIENTATION_CLASSIFY = False
PP_USE_DOC_UNWARPING = False
PP_USE_TEXTLINE_ORIENTATION = False
PP_USE_TABLE_RECOGNITION = True
PP_USE_FORMULA_RECOGNITION = False
PP_USE_CHART_RECOGNITION = False
PP_USE_SEAL_RECOGNITION = False
PP_CONFIG_PATH = ''  # optional PP-StructureV3 yaml/config path
PP_TABLE_TEMP_DIR = '/kaggle/working/pp_table_crops'
MAX_PIXELS_TABLE_PP = 1_200_000
PP_HINT_MAX_CHARS = 2500

# Output cleanup keeps OCR faithful: normalize notation, but do not fix spelling.
NORMALIZE_OCR_OUTPUT = True
NORMALIZE_INLINE_WHITESPACE = True
NORMALIZE_LATIN_CYRILLIC_LOOKALIKES = False  # risky for formulas/Latin names; enable only after validation
APPLY_VISIBLE_STRIKETHROUGH_CORRECTIONS = False  # True: ~~old~~{new} -> new and ~~text~~ -> text

SPECIAL_TEXT_MARKER_RULES = (
    'Use these special markers only when they are visible in the crop: '
    '~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, '
    'and [illegible] for an unreadable word inside an otherwise legible line. '
)

OUTPUT_NOTATION_RULES = (
    'Use stable plain-text notation for visible symbols: straight quotes, hyphen for dash variants, '
    '^ and _ for visible superscripts/subscripts, simple plain text for common LaTeX-like symbols, '
    'and compact whitespace. These notation choices must not change spelling, grammar, meaning, or content. '
)

DOCUMENT_CONTEXT_RULES = '''The crop may come from one of the following document sources:
- Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.
- Historical Ukrainian/Cyrillic archives and manuscripts. Preserve original spelling, punctuation, and orthography; do not modernize or normalize.
- School homework. It may contain corrections, teacher marks, formulas, diagrams, mixed handwriting and printed text.
- University exams, coursework, lecture notes, scientific documents, tables, formulas, chemistry notation, mathematics, and technical symbols.
General OCR rules:
- Read only what is visually present in the image.
- Do not infer, reconstruct, autocomplete, or guess missing text.
- Do not use memorized canonical versions of poems, dictations, historical texts, exercises, or formulas.
- Preserve original spelling, capitalization, punctuation, line breaks, and formatting whenever possible.
- Keep uncertain characters exactly as seen; do not silently correct them.
- Preserve crossed-out text, corrections, annotations, and teacher marks when visible.
- Output only the transcription of visible content.'''

COMMON_OCR_RULES = (
    'Return only the transcription. No JSON, no Markdown, no explanation. '
    + DOCUMENT_CONTEXT_RULES
    + SPECIAL_TEXT_MARKER_RULES
    + OUTPUT_NOTATION_RULES
    + 'Preserve punctuation, line content, corrections, spelling mistakes, capitalization, digits, '
    'abbreviations, quotes, hyphens, line-final dashes, and visible spacing as much as possible. '
    'Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, '
    'or infer hidden/missing text.'
)

CROP_PROMPTS = {
    'handwritten': (
        'Transcribe the visible handwritten text exactly. '
        'Preserve punctuation, line content, corrections, and strikethrough markers. '
        + COMMON_OCR_RULES
    ),
    'printed': (
        'Transcribe the visible printed or typed text exactly. '
        'Preserve punctuation, line content, corrections, and strikethrough markers. '
        + COMMON_OCR_RULES
    ),
    'annotation': (
        'Read this short annotation, teacher mark, grade, correction, or numbering. '
        'Return only the exact visible text. '
        + SPECIAL_TEXT_MARKER_RULES
        + OUTPUT_NOTATION_RULES
        + 'No explanation.'
    ),
    'formula': (
        'Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, '
        'or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the '
        'clearest representation and plain Unicode when it better matches the handwriting. Do not wrap the '
        'answer in dollar signs. Preserve visible symbols, indices, superscripts, subscripts, arrows, fractions, '
        'matrix/determinant structure, punctuation, numbering, and strikethrough/correction markers. '
        'Do not solve, simplify, normalize, explain, or convert old notation into a different style. '
        + OUTPUT_NOTATION_RULES
    ),
    'table': (
        'Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row '
        'and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, '
        'column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, visible spelling '
        'mistakes, corrections, and strikethrough markers. Do not infer missing cells, rebalance columns, summarize, '
        'or explain. '
        + OUTPUT_NOTATION_RULES
    ),
    'image': 'Return an empty string.',
    'graph': 'Return an empty string.',
    'default': 'Transcribe the visible content exactly. ' + COMMON_OCR_RULES,
}

TABLE_PP_HINT_INSTRUCTION = (
    'PP-StructureV3 structural hint for this same table crop is provided below. '
    'Use it only to understand row and column boundaries. Transcribe from the image itself; '
    'if the hint conflicts with the visible image, trust the image. Return only pipe-separated table text.'
)

VALID_TYPES = {'handwritten', 'printed', 'formula', 'table', 'annotation', 'image', 'graph'}
TEXT_TYPES = {'handwritten', 'printed', 'formula', 'table', 'annotation'}
IMAGE_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.webp', '.bmp']
TYPE_ALIASES = {
    'text': 'handwritten',
    'plain text': 'printed',
    'title': 'handwritten',
    'caption': 'printed',
    'isolate_formula': 'formula',
    'equation': 'formula',
    'tabular': 'table',
    'figure': 'image',
    'picture': 'image',
    'chart': 'graph',
}


In [ ]:
import subprocess
import sys
from pathlib import Path

def run_cmd(cmd):
    print('Running:', ' '.join(str(x) for x in cmd), flush=True)
    subprocess.check_call([str(x) for x in cmd])

def pip_install(packages, index_url=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed'] + list(packages)
    if index_url:
        cmd += ['--extra-index-url', index_url]
    run_cmd(cmd)

if INSTALL_DEPS:
    pip_install(['accelerate', 'peft', 'bitsandbytes', 'qwen-vl-utils', 'pandas==2.2.2', 'pillow<12', 'opencv-python-headless'])
    if INSTALL_TRANSFORMERS_FROM_GIT:
        pip_install(['git+https://github.com/huggingface/transformers.git'])

if INSTALL_DOCLAYOUT_YOLO:
    repo = Path(DOCLAYOUT_REPO)
    if not repo.exists():
        run_cmd(['git', 'clone', '--depth', '1', 'https://github.com/opendatalab/DocLayout-YOLO.git', str(repo)])
    pip_install(['-e', str(repo)])

if INSTALL_PADDLEOCR:
    if PADDLE_INSTALL_MODE == 'cpu':
        pip_install([PADDLEPADDLE_CPU_PACKAGE], PADDLE_CPU_INDEX_URL)
    elif PADDLE_INSTALL_MODE == 'gpu':
        pip_install([PADDLEPADDLE_GPU_PACKAGE], PADDLE_GPU_INDEX_URL)
    elif PADDLE_INSTALL_MODE != 'skip':
        raise ValueError(f'Unknown PADDLE_INSTALL_MODE={PADDLE_INSTALL_MODE}')
    pip_install([PADDLEOCR_PACKAGE])


In [ ]:
import contextlib
import gc
import hashlib
import json
import logging
import math
import os
import re
import subprocess
import time
import warnings
from pathlib import Path
from types import ModuleType

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

try:
    from torchvision.ops import nms as torch_nms
except Exception:
    torch_nms = None

Image.MAX_IMAGE_PIXELS = None
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
if PADDLE_PDX_MODEL_SOURCE:
    os.environ['PADDLE_PDX_MODEL_SOURCE'] = PADDLE_PDX_MODEL_SOURCE

def suppress_noise():
    warnings.filterwarnings('ignore', message=r'.*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*')
    logging.getLogger('transformers').setLevel(logging.ERROR)
    logging.getLogger('transformers.processing_utils').setLevel(logging.ERROR)
    try:
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
    except Exception:
        pass

suppress_noise()

def log_msg(message):
    if VERBOSE_LOG:
        print(f'[{time.strftime("%H:%M:%S")}] {message}', flush=True)

for repo_path in [DOCLAYOUT_REPO, '/kaggle/input/DocLayout-YOLO', '/kaggle/input/doclayout-yolo/DocLayout-YOLO']:
    p = Path(repo_path)
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

if 'doclayout_yolo.utils.callbacks.hub' not in sys.modules:
    dummy_hub = ModuleType('doclayout_yolo.utils.callbacks.hub')
    dummy_hub.callbacks = {}
    sys.modules['doclayout_yolo.utils.callbacks.hub'] = dummy_hub

def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith('/') and Path(item).exists():
            return item
        if not item.startswith('/'):
            return item
    raise FileNotFoundError('No Qwen3-VL base model found. Add a Kaggle input path or enable internet.')

def find_yolo_weights():
    for item in YOLO_WEIGHT_CANDIDATES:
        p = Path(item)
        if p.exists():
            return p
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for p in input_root.rglob('*.pt'):
            text = str(p).lower()
            if 'yolo' in text or 'doclayout' in text:
                return p
    raise FileNotFoundError('No YOLO .pt weights found. Update YOLO_WEIGHT_CANDIDATES.')

def find_lora_dir():
    if not USE_QWEN_LORA:
        return None
    for item in LORA_CANDIDATES:
        p = Path(item)
        if (p / 'adapter_config.json').exists():
            return p
    if LORA_FALLBACK_SEARCH:
        input_root = Path('/kaggle/input')
        if input_root.exists():
            candidates = []
            for root, _, files in os.walk(input_root):
                if 'adapter_config.json' in files:
                    root_path = Path(root)
                    low = str(root_path).lower()
                    score = sum(token in low for token in ('ukrainian', 'curriculum', 'stage2', 'final', 'rukopys', 'qwen3vl'))
                    score += 3 * int('qwen3vl_ukrainian_stage2_final_lora_final' in low)
                    score -= 3 * sum(token in low for token in ('stage0', 'stage1', 'silver'))
                    candidates.append((score, root_path))
            if candidates:
                return sorted(candidates, key=lambda x: (-x[0], str(x[1])))[0][1]
    raise FileNotFoundError('No Qwen LoRA adapter_config.json found. Add the Ukrainian Stage 2 final LoRA Kaggle input or set USE_QWEN_LORA=False.')

def load_prompt_config(lora_path):
    global CROP_PROMPTS, MAX_PIXELS_CROP
    if lora_path is None:
        return
    cfg_path = Path(lora_path) / 'rukopys_prompt_config.json'
    if not cfg_path.exists():
        return
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    if LOAD_LORA_CROP_PROMPTS:
        CROP_PROMPTS.update(cfg.get('crop_prompts', {}))
    MAX_PIXELS_CROP = int(cfg.get('max_pixels_crop', MAX_PIXELS_CROP))

def get_dataset_root():
    root = Path(DATASET_ROOT)
    split = 'train' if RUN_SPLIT == 'validation' else 'test'
    if not (root / split / 'metadata.jsonl').exists():
        raise FileNotFoundError(f'Expected {root / split / "metadata.jsonl"}')
    return root

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def resolve_image_path(root, split, file_name):
    raw = Path(file_name)
    names = [raw.name] + [raw.stem + ext for ext in IMAGE_EXTENSIONS if raw.stem + ext != raw.name]
    candidates = [root / split / file_name, root / file_name]
    for name in names:
        candidates.extend([root / split / 'images' / name, root / split / name])
    for p in candidates:
        if p.exists():
            return str(p)
    return str(root / split / 'images' / names[0])

def normalize_type(value):
    value = str(value or 'handwritten').strip().lower().replace('_', ' ')
    value = TYPE_ALIASES.get(value, value)
    if value in VALID_TYPES:
        return value
    if 'table' in value:
        return 'table'
    if 'formula' in value or 'equation' in value:
        return 'formula'
    if 'print' in value:
        return 'printed'
    if 'annot' in value or 'mark' in value:
        return 'annotation'
    if 'image' in value or 'figure' in value or 'picture' in value:
        return 'image'
    if 'graph' in value or 'chart' in value:
        return 'graph'
    return 'handwritten'

def clamp_xyxy(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]

def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(1, area_a + area_b - inter)

def python_nms(boxes, scores, threshold):
    order = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        cur = order.pop(0)
        keep.append(cur)
        order = [idx for idx in order if iou(boxes[cur], boxes[idx]) <= threshold]
    return keep

def sort_regions(regions):
    return sorted(regions, key=lambda r: (r['bbox'][1], r['bbox'][0]))

def strip_internal_fields(region):
    return {'bbox': region['bbox'], 'type': normalize_type(region.get('type')), 'text': str(region.get('text') or '')}

QUOTE_TRANSLATION = str.maketrans({
    chr(0x2018): chr(39), chr(0x2019): chr(39), chr(0x201A): chr(39), chr(0x201B): chr(39), chr(0x02BC): chr(39),
    chr(0x201C): chr(34), chr(0x201D): chr(34), chr(0x201E): chr(34), chr(0x201F): chr(34),
    chr(0x00AB): chr(34), chr(0x00BB): chr(34),
})
DASH_TRANSLATION = str.maketrans({
    chr(0x2010): '-', chr(0x2011): '-', chr(0x2012): '-', chr(0x2013): '-', chr(0x2014): '-', chr(0x2212): '-',
})
SUPERSCRIPT_TRANSLATION = {
    chr(0x2070): '0', chr(0x00B9): '1', chr(0x00B2): '2', chr(0x00B3): '3', chr(0x2074): '4',
    chr(0x2075): '5', chr(0x2076): '6', chr(0x2077): '7', chr(0x2078): '8', chr(0x2079): '9',
    chr(0x207A): '+', chr(0x207B): '-', chr(0x207C): '=', chr(0x207D): '(', chr(0x207E): ')',
}
SUBSCRIPT_TRANSLATION = {
    chr(0x2080): '0', chr(0x2081): '1', chr(0x2082): '2', chr(0x2083): '3', chr(0x2084): '4',
    chr(0x2085): '5', chr(0x2086): '6', chr(0x2087): '7', chr(0x2088): '8', chr(0x2089): '9',
    chr(0x208A): '+', chr(0x208B): '-', chr(0x208C): '=', chr(0x208D): '(', chr(0x208E): ')',
}
LATEX_SYMBOL_NORMALIZATIONS = {
    r'\\Rightarrow': '=>', r'\\rightarrow': '->', r'\\to': '->',
    r'\\Leftarrow': '<=', r'\\leftarrow': '<-', r'\\leftrightarrow': '<->',
    r'\\cdot': chr(0x00B7), r'\\times': chr(0x00D7), r'\\pm': '+/-',
    r'\\leq': '<=', r'\\geq': '>=', r'\\neq': '!=', r'\\infty': 'inf',
    r'\\alpha': chr(0x03B1), r'\\beta': chr(0x03B2), r'\\gamma': chr(0x03B3), r'\\delta': chr(0x03B4),
    r'\\theta': chr(0x03B8), r'\\lambda': chr(0x03BB), r'\\mu': chr(0x03BC), r'\\pi': chr(0x03C0),
    r'\\sigma': chr(0x03C3), r'\\omega': chr(0x03C9), r'\\Delta': chr(0x0394), r'\\Omega': chr(0x03A9),
}
CYRILLIC_LOOKALIKE_TRANSLATION = str.maketrans({
    'c': chr(0x0441), 'o': chr(0x043E), 'p': chr(0x0440), 'x': chr(0x0445),
    'C': chr(0x0421), 'O': chr(0x041E), 'P': chr(0x0420), 'X': chr(0x0425),
})

def has_cyrillic(text):
    return any(0x0400 <= ord(ch) <= 0x04FF for ch in str(text or ''))

def normalize_script_digits(text):
    out = []
    active = None
    for ch in text:
        if ch in SUPERSCRIPT_TRANSLATION:
            if active != '^':
                out.append('^')
                active = '^'
            out.append(SUPERSCRIPT_TRANSLATION[ch])
        elif ch in SUBSCRIPT_TRANSLATION:
            if active != '_':
                out.append('_')
                active = '_'
            out.append(SUBSCRIPT_TRANSLATION[ch])
        else:
            out.append(ch)
            active = None
    return ''.join(out)

def normalize_latex_symbols(text):
    for pattern, repl in LATEX_SYMBOL_NORMALIZATIONS.items():
        text = re.sub(pattern + r'\b', repl, text)
    text = re.sub(r'(?<=\w)\^\{([^{}\n]{1,16})\}', r'^\1', text)
    text = re.sub(r'(?<=\w)_\{([^{}\n]{1,16})\}', r'_\1', text)
    return text

def normalize_latin_cyrillic_lookalikes(text):
    def fix_token(match):
        token = match.group(0)
        if has_cyrillic(token):
            return token.translate(CYRILLIC_LOOKALIKE_TRANSLATION)
        return token
    return re.sub(r'\S+', fix_token, text)

def normalize_strikethrough_markers(text):
    if APPLY_VISIBLE_STRIKETHROUGH_CORRECTIONS:
        text = re.sub(r'~~([^~{}\n]+)~~\{([^{}\n]*)\}', r'\2', text)
        text = re.sub(r'~~([^~\n]+)~~', r'\1', text)
    return text

def normalize_output_whitespace(text):
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', line).strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def normalize_ocr_output_text(text):
    text = str(text or '')
    if not NORMALIZE_OCR_OUTPUT:
        return text.strip()
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = normalize_strikethrough_markers(text)
    text = text.translate(QUOTE_TRANSLATION).translate(DASH_TRANSLATION)
    text = normalize_latex_symbols(text)
    text = normalize_script_digits(text)
    if NORMALIZE_LATIN_CYRILLIC_LOOKALIKES:
        text = normalize_latin_cyrillic_lookalikes(text)
    if NORMALIZE_INLINE_WHITESPACE:
        return normalize_output_whitespace(text)
    return text.strip()

def clean_crop_text(text, max_chars=MAX_REGION_TEXT_CHARS):
    text = (text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```[a-zA-Z]*', '', text).strip()
        text = re.sub(r'```$', '', text).strip()
    text = re.sub(r'^(text|transcription|answer)\s*:\s*', '', text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {chr(34), chr(39)}:
        text = text[1:-1].strip()
    if text.startswith('[') or text.startswith('{'):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and 'text' in obj:
                text = str(obj['text'])
            else:
                return ''
        except Exception:
            return ''
    return normalize_ocr_output_text(text)[:max_chars]

def resize_to_pixel_budget(img, max_pixels):
    w, h = img.size
    total = max(1, w * h)
    if total <= max_pixels:
        return img
    scale = (max_pixels / total) ** 0.5
    return img.resize((max(1, int(round(w * scale))), max(1, int(round(h * scale)))), Image.Resampling.LANCZOS)

def crop_image(image_path, bbox, max_pixels=MAX_PIXELS_CROP):
    with Image.open(image_path) as img:
        img = img.convert('RGB')
        w, h = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * CROP_PAD_RATIO))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        return resize_to_pixel_budget(img.crop((x1, y1, x2, y2)), max_pixels)

model_id = find_model_id()
lora_dir = find_lora_dir()
yolo_weights = find_yolo_weights()
dataset_root = get_dataset_root()
load_prompt_config(lora_dir)
IMAGE_SPLIT = 'train' if RUN_SPLIT == 'validation' else 'test'
if RUN_SPLIT == 'validation':
    records = read_jsonl(dataset_root / 'train' / 'metadata.jsonl')
    OUTPUT_CSV = 'validation_pred.csv'
else:
    records = read_jsonl(dataset_root / 'test' / 'metadata.jsonl')
if TEST_MODE:
    records = records[:TEST_LIMIT]
print('Base model:', model_id)
print('Qwen LoRA:', lora_dir if lora_dir is not None else 'disabled')
print('YOLO weights:', yolo_weights)
print('Dataset:', dataset_root)
print('Images:', len(records))


In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

def qwen_device():
    if QWEN_DEVICE != 'auto':
        return QWEN_DEVICE
    return 'cuda:0' if torch.cuda.is_available() else 'cpu'

def yolo_predict_device(qwen_dev):
    if YOLO_DEVICE != 'auto':
        return YOLO_DEVICE
    if str(qwen_dev).startswith('cuda'):
        parts = str(qwen_dev).split(':')
        return int(parts[1]) if len(parts) > 1 else 0
    return 'cpu'

def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = 'left'
    return processor

def load_qwen_model(device):
    is_cuda = str(device).startswith('cuda')
    kwargs = {
        'device_map': {'': device},
        'trust_remote_code': True,
        'low_cpu_mem_usage': True,
    }
    if QWEN_ATTN_IMPLEMENTATION:
        kwargs['attn_implementation'] = QWEN_ATTN_IMPLEMENTATION
    if QWEN_LOAD_IN_4BIT and is_cuda:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
        kwargs['dtype'] = torch.float16
    else:
        kwargs['dtype'] = torch.float16 if is_cuda else torch.float32
    try:
        model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
    except TypeError:
        kwargs['torch_dtype'] = kwargs.pop('dtype')
        model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
    if lora_dir is not None:
        print(f'Loading Qwen LoRA adapter: {lora_dir}', flush=True)
        model = PeftModel.from_pretrained(model, str(lora_dir))
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    return model, processor

def apply_chat_template(processor, messages):
    candidates = [
        {'tokenize': False, 'add_generation_prompt': True, 'template_kwargs': {'enable_thinking': False}},
        {'tokenize': False, 'add_generation_prompt': True, 'processor_kwargs': {'enable_thinking': False}},
        {'tokenize': False, 'add_generation_prompt': True, 'enable_thinking': False},
        {'tokenize': False, 'add_generation_prompt': True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', message=r'.*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*')
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    processor.tokenizer.padding_side = 'left'
    texts = [apply_chat_template(processor, m) for m in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={'padding': True, 'return_tensors': 'pt'},
            images_kwargs={'return_tensors': 'pt'},
            videos_kwargs={'return_tensors': 'pt'},
        )
    except TypeError:
        inputs = processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt')
    inputs = inputs.to(device)
    amp = torch.amp.autocast('cuda', dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()
    with torch.no_grad(), amp:
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return decoded

def get_yolo_names(yolo_model):
    names = getattr(yolo_model, 'names', None)
    if names is None and hasattr(yolo_model, 'model'):
        names = getattr(yolo_model.model, 'names', None)
    return names or {}

def get_class_name(names, cls_id):
    if isinstance(names, dict):
        return names.get(cls_id, names.get(str(cls_id), 'handwritten'))
    if isinstance(names, (list, tuple)) and 0 <= cls_id < len(names):
        return names[cls_id]
    return 'handwritten'

def load_yolo_model(device):
    errors = []
    if YOLO_BACKEND in {'auto', 'doclayout_yolo'}:
        try:
            from doclayout_yolo import YOLOv10
            model = YOLOv10(str(yolo_weights))
            try:
                model.to(device)
            except Exception as e:
                print('YOLO .to(device) skipped:', e, flush=True)
            print('YOLO backend: doclayout_yolo')
            return model
        except Exception as e:
            errors.append(f'doclayout_yolo: {e}')
    if YOLO_BACKEND in {'auto', 'ultralytics'}:
        try:
            from ultralytics import YOLO
            model = YOLO(str(yolo_weights))
            print('YOLO backend: ultralytics')
            return model
        except Exception as e:
            errors.append(f'ultralytics: {e}')
    raise RuntimeError('Could not load YOLO. ' + ' | '.join(errors))

def dedupe_yolo_regions(regions):
    kept = []
    for region in sorted(regions, key=lambda r: float(r.get('_score', 0.0)), reverse=True):
        if not any(iou(region['bbox'], old['bbox']) > YOLO_DEDUP_IOU for old in kept):
            kept.append(region)
    return sort_regions([strip_internal_fields(r) for r in kept])

def postprocess_yolo_result(result, img_w, img_h, names):
    boxes, scores, labels = [], [], []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        boxes.append([x1, y1, x2, y2])
        scores.append(float(box.conf[0]))
        labels.append(int(box.cls[0]))
    if not boxes:
        return []
    keep_indices = []
    for cls_id in sorted(set(labels)):
        cls_indices = [i for i, label in enumerate(labels) if label == cls_id]
        cls_boxes = [boxes[i] for i in cls_indices]
        cls_scores = [scores[i] for i in cls_indices]
        if torch_nms is not None:
            kept = torch_nms(torch.tensor(cls_boxes, dtype=torch.float32), torch.tensor(cls_scores, dtype=torch.float32), YOLO_IOU_NMS).tolist()
        else:
            kept = python_nms(cls_boxes, cls_scores, YOLO_IOU_NMS)
        keep_indices.extend([cls_indices[i] for i in kept])
    regions = []
    for idx in sorted(set(keep_indices)):
        x1, y1, x2, y2 = boxes[idx]
        h = max(1.0, y2 - y1)
        box = clamp_xyxy([x1 - YOLO_PAD_SCALE_X * h, y1 - YOLO_PAD_SCALE_Y * h, x2 + YOLO_PAD_SCALE_X * h, y2 + YOLO_PAD_SCALE_Y * h], img_w, img_h)
        if box is None:
            continue
        regions.append({'bbox': box, 'type': normalize_type(get_class_name(names, labels[idx])), 'text': '', '_score': scores[idx]})
    return dedupe_yolo_regions(regions)

def detect_regions_yolo(yolo_model, image_path, yolo_device):
    results = yolo_model.predict(source=str(image_path), imgsz=YOLO_IMG_SIZE, conf=YOLO_CONF, max_det=YOLO_MAX_DET, verbose=False, device=yolo_device)
    result = results[0]
    if hasattr(result, 'orig_shape') and result.orig_shape:
        img_h, img_w = result.orig_shape
    else:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
    return postprocess_yolo_result(result, img_w, img_h, get_yolo_names(yolo_model))


In [ ]:
def load_pp_structure():
    if not PP_ENABLE:
        return None
    from paddleocr import PPStructureV3
    init_candidates = []
    base = {
        'device': PP_DEVICE,
        'lang': PP_LANG,
        'use_doc_orientation_classify': PP_USE_DOC_ORIENTATION_CLASSIFY,
        'use_doc_unwarping': PP_USE_DOC_UNWARPING,
        'use_textline_orientation': PP_USE_TEXTLINE_ORIENTATION,
    }
    if PP_CONFIG_PATH:
        with_config = dict(base)
        with_config['paddlex_config'] = PP_CONFIG_PATH
        init_candidates.append(with_config)
    init_candidates.extend([base, {'device': PP_DEVICE, 'lang': PP_LANG}, {'lang': PP_LANG}, {}])
    last_error = None
    for kwargs in init_candidates:
        try:
            pipe = PPStructureV3(**kwargs)
            print('Loaded PP-StructureV3 with args:', kwargs)
            return pipe
        except TypeError as e:
            last_error = e
            continue
    raise RuntimeError(f'Could not initialize PPStructureV3: {last_error}')

def pp_crop_path(image_path, bbox):
    Path(PP_TABLE_TEMP_DIR).mkdir(parents=True, exist_ok=True)
    key = f'{Path(image_path).stem}_{bbox}'.encode('utf-8')
    name = hashlib.md5(key).hexdigest()[:16] + '.png'
    return Path(PP_TABLE_TEMP_DIR) / name

def normalize_pp_text(text):
    text = str(text or '').strip()
    text = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def truncate_text(text, max_chars):
    text = str(text or '').strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + '\n[truncated]'

def markdown_from_result(res):
    md = getattr(res, 'markdown', None)
    if not md:
        return ''
    if isinstance(md, dict):
        text = md.get('markdown_texts') or md.get('markdown_text') or md.get('text') or ''
        if isinstance(text, list):
            text = '\n'.join(str(x) for x in text)
        return normalize_pp_text(text)
    return normalize_pp_text(md)

def collect_table_strings(obj, out):
    if isinstance(obj, dict):
        for value in obj.values():
            collect_table_strings(value, out)
    elif isinstance(obj, list):
        for value in obj:
            collect_table_strings(value, out)
    elif isinstance(obj, str):
        s = normalize_pp_text(obj)
        low = s.lower()
        if '<table' in low or ('|' in s and '\n' in s):
            out.append(s)

def json_excerpt(obj):
    def prune(x):
        if isinstance(x, dict):
            skip = {'input_path', 'page_index', 'img', 'image', 'output_img', 'markdown_images'}
            return {k: prune(v) for k, v in x.items() if k not in skip}
        if isinstance(x, list):
            return [prune(v) for v in x[:20]]
        return x
    return truncate_text(json.dumps(prune(obj), ensure_ascii=False, default=str), PP_HINT_MAX_CHARS)

def run_pp_table_hint(pp_pipeline, image_path, bbox):
    if pp_pipeline is None:
        return ''
    t0 = time.time()
    log_msg(f'PP table hint start: {Path(image_path).name} bbox={bbox}')
    path = pp_crop_path(image_path, bbox)
    if not path.exists():
        crop_image(image_path, bbox, MAX_PIXELS_TABLE_PP).save(path)
    log_msg(f'PP predict start: {path.name}')
    predict_kwargs = {
        'use_doc_orientation_classify': PP_USE_DOC_ORIENTATION_CLASSIFY,
        'use_doc_unwarping': PP_USE_DOC_UNWARPING,
        'use_textline_orientation': PP_USE_TEXTLINE_ORIENTATION,
        'use_table_recognition': PP_USE_TABLE_RECOGNITION,
        'use_formula_recognition': PP_USE_FORMULA_RECOGNITION,
        'use_chart_recognition': PP_USE_CHART_RECOGNITION,
        'use_seal_recognition': PP_USE_SEAL_RECOGNITION,
    }
    try:
        output = list(pp_pipeline.predict(input=str(path), **predict_kwargs))
    except TypeError:
        output = list(pp_pipeline.predict(input=str(path)))
    log_msg(f'PP predict done: {path.name}, results={len(output)}, elapsed={time.time() - t0:.1f}s')
    pieces = []
    table_strings = []
    json_pieces = []
    for res in output:
        md = markdown_from_result(res)
        if md:
            pieces.append('Markdown/table text:\n' + md)
        js = getattr(res, 'json', None)
        if js:
            collect_table_strings(js, table_strings)
            json_pieces.append(js)
    if table_strings:
        pieces.append('Table-like strings:\n' + '\n\n'.join(table_strings[:3]))
    if not pieces and json_pieces:
        pieces.append('Structured JSON excerpt:\n' + json_excerpt(json_pieces[0]))
    hint = truncate_text('\n\n'.join(pieces), PP_HINT_MAX_CHARS)
    if LOG_TABLE_HINT_PREVIEW and hint:
        print(hint[:PP_HINT_PREVIEW_CHARS], flush=True)
    log_msg(f'PP table hint done: chars={len(hint)}, elapsed={time.time() - t0:.1f}s')
    return hint

def table_prompt_with_hint(hint):
    if not hint:
        return CROP_PROMPTS['table']
    return CROP_PROMPTS['table'] + '\n\n' + TABLE_PP_HINT_INSTRUCTION + '\n\n' + hint


In [ ]:
def should_crop_ocr(region):
    if CROP_OCR_MODE == 'none':
        return False
    if region.get('type') not in TEXT_TYPES:
        return False
    if CROP_OCR_MODE == 'all_text':
        return True
    text = str(region.get('text') or '')
    return (not text.strip()) or len(text) < 4 or len(text) > 160

def crop_messages(image_path, region, pp_pipeline):
    rtype = normalize_type(region.get('type'))
    max_pixels = MAX_PIXELS_TABLE_QWEN if rtype == 'table' else MAX_PIXELS_CROP
    prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS['default'])
    if rtype == 'table' and PP_ENABLE:
        try:
            hint = run_pp_table_hint(pp_pipeline, image_path, region['bbox'])
            prompt = table_prompt_with_hint(hint)
        except Exception as e:
            print('PP-StructureV3 table hint failed:', e, flush=True)
    return [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': crop_image(image_path, region['bbox'], max_pixels)},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]

def ocr_regions(qwen_model, processor, image_path, regions, device, pp_pipeline):
    crop_indices = [i for i, r in enumerate(regions) if should_crop_ocr(r)]
    log_msg(f'OCR start: {Path(image_path).name}, crops={len(crop_indices)}/{len(regions)}')
    for start in range(0, len(crop_indices), CROP_BATCH_SIZE):
        batch_t0 = time.time()
        batch_indices = crop_indices[start:start + CROP_BATCH_SIZE]
        batch_types = [normalize_type(regions[i].get('type')) for i in batch_indices]
        batch_no = start // max(1, CROP_BATCH_SIZE) + 1
        total_batches = math.ceil(len(crop_indices) / max(1, CROP_BATCH_SIZE))
        log_msg(f'Qwen batch prepare {batch_no}/{total_batches}: idx={batch_indices}, types={batch_types}')
        msgs = [crop_messages(image_path, regions[i], pp_pipeline) for i in batch_indices]
        max_tokens = max(MAX_NEW_TOKENS_TABLE if normalize_type(regions[i].get('type')) == 'table' else MAX_NEW_TOKENS_CROP for i in batch_indices)
        log_msg(f'Qwen generate start {batch_no}/{total_batches}: max_tokens={max_tokens}')
        try:
            outs = generate_batch(qwen_model, processor, msgs, device, max_tokens)
        except Exception as e:
            print('Qwen OCR batch failed:', e, flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            continue
        log_msg(f'Qwen generate done {batch_no}/{total_batches}: elapsed={time.time() - batch_t0:.1f}s')
        for idx, out in zip(batch_indices, outs):
            text = clean_crop_text(out)
            if text:
                regions[idx]['text'] = text
    cleaned = sort_regions([strip_internal_fields(r) for r in regions])
    log_msg(f'OCR done: {Path(image_path).name}, regions={len(cleaned)}')
    return cleaned

def infer_one_image(qwen_model, processor, yolo_model, pp_pipeline, image_path, device, yolo_device):
    t0 = time.time()
    log_msg(f'YOLO start: {Path(image_path).name}')
    regions = detect_regions_yolo(yolo_model, image_path, yolo_device)
    counts = pd.Series([r.get('type') for r in regions]).value_counts().to_dict() if regions else {}
    log_msg(f'YOLO done: {Path(image_path).name}, regions={len(regions)}, types={counts}, elapsed={time.time() - t0:.1f}s')
    if not regions:
        return []
    regions = ocr_regions(qwen_model, processor, image_path, regions, device, pp_pipeline)
    log_msg(f'Image inference done: {Path(image_path).name}, elapsed={time.time() - t0:.1f}s')
    return regions

print('Inference helpers ready. Models will be loaded inside each worker.')


In [ ]:
import multiprocessing as mp

def detect_cuda_count_without_torch():
    try:
        out = subprocess.check_output(['nvidia-smi', '-L'], stderr=subprocess.STDOUT).decode('utf-8')
        return len([line for line in out.splitlines() if line.strip().startswith('GPU ')])
    except Exception:
        return 0

def available_gpu_ids():
    count = detect_cuda_count_without_torch()
    if count <= 0:
        return [None]
    requested = [int(x) for x in GPU_IDS]
    ids = [x for x in requested if 0 <= x < count]
    if not ids:
        ids = list(range(min(2, count)))
    if not MULTI_GPU:
        ids = ids[:1]
    return ids

def split_records_round_robin(items, n):
    return [items[i::n] for i in range(n)]

def save_results_csv(results, output_csv):
    tmp = output_csv + '.tmp'
    pd.DataFrame(results).drop_duplicates(subset=['image'], keep='last').to_csv(tmp, index=False)
    os.replace(tmp, output_csv)

def worker_process(gpu_id, worker_records, output_csv):
    suppress_noise()
    label = 'CPU' if gpu_id is None else f'GPU {gpu_id}'
    device = 'cpu' if gpu_id is None else f'cuda:{gpu_id}'
    yolo_device = 'cpu' if gpu_id is None else gpu_id
    print(f'[{label}] loading models for {len(worker_records)} images', flush=True)
    qwen_model, processor = load_qwen_model(device)
    yolo_model = load_yolo_model(device)
    pp_pipeline = load_pp_structure()
    print(f'[{label}] models loaded', flush=True)

    done = set()
    results = []
    if RESUME_OUTPUT and Path(output_csv).exists():
        try:
            old = pd.read_csv(output_csv).drop_duplicates(subset=['image'], keep='last')
            done = set(old['image'].tolist())
            results = old.to_dict('records')
            print(f'[{label}] resumed {len(done)}/{len(worker_records)} rows from {output_csv}', flush=True)
        except Exception as e:
            print(f'[{label}] could not resume {output_csv}: {e}', flush=True)

    run_start = time.time()
    processed_this_run = 0
    for pos, rec in enumerate(worker_records, start=1):
        image_name = Path(rec['file_name']).name
        if image_name in done:
            continue
        image_path = resolve_image_path(dataset_root, IMAGE_SPLIT, rec['file_name'])
        image_t0 = time.time()
        try:
            regions = infer_one_image(qwen_model, processor, yolo_model, pp_pipeline, image_path, device, yolo_device)
        except Exception as e:
            print(f'[{label}] failed {image_name}: {e}', flush=True)
            regions = []
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
        results.append({'image': image_name, 'regions': json.dumps(regions, ensure_ascii=False)})
        done.add(image_name)
        processed_this_run += 1
        if LOG_IMAGE_DONE_EVERY <= 1 or processed_this_run % LOG_IMAGE_DONE_EVERY == 0:
            elapsed = time.time() - image_t0
            print(f'[{label}] done {len(done)}/{len(worker_records)}: {image_name}, regions={len(regions)}, elapsed={elapsed:.1f}s', flush=True)
        if len(results) % CHECKPOINT_EVERY == 0:
            save_results_csv(results, output_csv)
            elapsed = max(1e-6, time.time() - run_start)
            speed = processed_this_run / elapsed if processed_this_run else 0.0
            print(f'[{label}] checkpoint {len(done)}/{len(worker_records)} -> {output_csv}, speed={speed:.3f} img/s', flush=True)

    save_results_csv(results, output_csv)
    print(f'[{label}] finished {len(done)}/{len(worker_records)} -> {output_csv}', flush=True)

gpu_ids = available_gpu_ids()
chunks = split_records_round_robin(records, len(gpu_ids))
partial_paths = [f'{PARTIAL_PREFIX}{gpu_id if gpu_id is not None else "cpu"}.csv' for gpu_id in gpu_ids]
print('Workers:', list(zip(gpu_ids, [len(c) for c in chunks], partial_paths)), flush=True)

if len(gpu_ids) == 1:
    worker_process(gpu_ids[0], chunks[0], partial_paths[0])
else:
    mp.set_start_method('fork', force=True)
    processes = []
    for gpu_id, chunk, partial_path in zip(gpu_ids, chunks, partial_paths):
        p = mp.Process(target=worker_process, args=(gpu_id, chunk, partial_path))
        p.start()
        processes.append(p)
    for p in processes:
        p.join()
    bad = [p.exitcode for p in processes if p.exitcode not in (0, None)]
    if bad:
        raise RuntimeError(f'One or more workers failed: {bad}')

frames = []
for path in partial_paths:
    if Path(path).exists():
        frame = pd.read_csv(path)
        print(f'Partial {path}: rows={len(frame)}', flush=True)
        frames.append(frame)
if not frames:
    raise RuntimeError('No partial outputs were created.')

final = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['image'], keep='last')
order = [Path(r['file_name']).name for r in records]
final = final.set_index('image').reindex(order).reset_index()
final['regions'] = final['regions'].fillna('[]')
final.to_csv(OUTPUT_CSV, index=False)
print('Wrote', OUTPUT_CSV, 'rows=', len(final), flush=True)
final.head()
